# Tutorial - Abstraction Mapping
This tutorial explains abstraction mapping: What it is and how to use it.

When deriving an explicit MDP from a simulator environment, it is most often the case that our original environment (simulator) has a multidimensional continuous state space and we want to discretize that space.
The means we establish a mapping to a (multidemensional) discrete space, which we call the abstract space.

Verigym makes it easy to define a mapping between the original environment and the abstracted environment by using an `AbstractionMapper`.
This class can hold two `AbstractionMap`s; one for the state space and one for the action space.
We demonstrate the creation and usage below.

The tutorial is structured as follows:

1. Gym spaces
2. Bin edges: cutting a continuous space into pieces
3. Different discretization types (linspace, polynomial, custom, data-based)
4. From bin edges to an `AbstractionMap`
5. Factored output vs. `original_to_enum`
6. Combining two maps into an `AbstractionMapper`
7. Backward kind: a point or a whole interval
8. Identity mapping
9. Putting it all together on a real environment

In [1]:
import functools

import gymnasium as gym
import numpy as np

import verigym
from verigym.abstraction.abstractionmapper import AbstractionMap, AbstractionMapper

## 1. Gym Spaces
`gymnasium` has a set of classes for standard types of state and action spaces.
The two we care about most are
- `gym.spaces.Box` for real numbers and
- `gym.spaces.MultiDiscrete` for integer numbers.

In [2]:
# 1-dimensional space on the interval [-1,1]
box_space = gym.spaces.Box(low=-1, high=1)
print(box_space)

# 1-dimensional space for natural numbers from {0, ..., 2}
discrete_space = gym.spaces.MultiDiscrete([3])
print(discrete_space)

Box(-1.0, 1.0, (1,), float32)
MultiDiscrete([3])


The whole point of an abstraction is to go from the first (uncountably many elements)
to the second (finitely many elements). `AbstractionMap` stores how many elements each side has.

In [3]:
from verigym.abstraction.gym_utils.spaces import get_n_elements_of_space

print("elements in the Box space:            ", get_n_elements_of_space(box_space))
print("elements in the MultiDiscrete space:  ", get_n_elements_of_space(discrete_space))

elements in the Box space:             inf
elements in the MultiDiscrete space:   3


## 2. Bin edges: discretizing a continuous space

Before we can map anything, we have to decide *where* to cut the continuous space.
These points are called **bin edges**. In VeriGym they are stored in a `BinEdges` object,
which holds one array of bin edges per dimension of the space.

The easiest way to create one is `generate_box_bins(space, bin_func, n_bins)`:

- `space`: the space we want to discretize (`Box`, `Discrete` or `MultiDiscrete`),
- `bin_func`: a function `(low, high, n_bins) -> array of edges`, e.g. `np.linspace`,
- `n_bins`: how many *representative values* to use per dimension.

A value is always mapped to the *closest bin edge below or equal to it*.
So `n_bins=5` means every dimension is described by 5 possible values. One consequence of
this is worth flagging early: the very top value (index `n_bins - 1`) is reached only by a
sample that lands exactly on `space.high` for that dimension -- we come back to this in
section 7, where it matters for interval backward maps.

In [4]:
from verigym.abstraction.discretization import generate_box_bins

# A 2-dimensional space: dimension 0 lives in [-1, 1], dimension 1 lives in [0, 10].
space = gym.spaces.Box(low=np.array([-1.0, 0.0], dtype=np.float32), high=np.array([1.0, 10.0], dtype=np.float32))

bin_edges = generate_box_bins(space, np.linspace, 5)

print("edges of dimension 0:", bin_edges[0])
print("edges of dimension 1:", bin_edges[1])
print("bins per dimension:  ", bin_edges.lengths)
print("nvec (shape of the discrete space):", bin_edges.nvec)

edges of dimension 0: [-1.  -0.5  0.   0.5  1. ]
edges of dimension 1: [ 0.   2.5  5.   7.5 10. ]
bins per dimension:   [5 5]
nvec (shape of the discrete space): [5 5]


We can now discretize a sample. `BinEdges` offers a family of codec methods between the four
representations of a sample: the original value (O), the value snapped onto a bin edge (V),
the per-dimension bin index (I), and a single enumerated integer (E). For now we only need
`orig_to_value` (O -> V) and `orig_to_idx` (O -> I).

In [5]:
sample = np.array([0.3, 7.4])
print("original sample: ", sample)
print("bin edge values: ", bin_edges.orig_to_value(sample))
print("bin edge indices:", bin_edges.orig_to_idx(sample))

original sample:  [0.3 7.4]


bin edge values:  [0. 5.]
bin edge indices: [2 2]


Note that we do not have to use the identical number of bins for each dimension.
If `n_bins` is an array (with the same shape as the space), each dimension gets its own resolution.
This is useful when one dimension matters more than another.

In [6]:
bin_edges_uneven = generate_box_bins(space, np.linspace, np.array([3, 6]))
print("edges of dimension 0:", bin_edges_uneven[0])
print("edges of dimension 1:", bin_edges_uneven[1])

edges of dimension 0: [-1.  0.  1.]
edges of dimension 1: [ 0.  2.  4.  6.  8. 10.]


## 3. Different discretization types

`np.linspace` puts the bin edges at equal distances. That may be a good default,
not always the best choice: maybe we care a lot about small values and barely about large ones.
Since `generate_box_bins` accepts *any* function `(low, high, n_bins) -> array`,
we can freely change how the space is cut.

### 3.1 Uniform bins (`np.linspace`)
Every bin has the same width. Use it when all regions of the space are equally important.

In [7]:
one_dim_space = gym.spaces.Box(low=0.0, high=10.0, shape=(1,))

linspace_bins = generate_box_bins(one_dim_space, np.linspace, 7)
print("linspace:      ", np.round(linspace_bins[0], 2))

linspace:       [ 0.    1.67  3.33  5.    6.67  8.33 10.  ]


### 3.2 Polynomial bins (`centered_pow_bin`)
`centered_pow_bin` places the edges according to a polynomial. The bins are narrow around
the center of the interval and wide towards the borders. Use it when the interesting
behaviour happens in the middle of the interval (e.g. a pole angle around 0).

The `power` argument is a keyword argument, so we fix it with `functools.partial`
to get a function with the required `(low, high, n_bins)` signature.

In [8]:
from verigym.abstraction.discretization import centered_pow_bin

pow_bins = generate_box_bins(one_dim_space, centered_pow_bin, 7)
print("centered_pow (power=2):", np.round(pow_bins[0], 2))

pow4_bin_func = functools.partial(centered_pow_bin, power=4)
pow4_bins = generate_box_bins(one_dim_space, pow4_bin_func, 7)
print("centered_pow (power=4):", np.round(pow4_bins[0], 2))

centered_pow (power=2): [ 0.    2.78  4.44  5.    5.56  7.22 10.  ]
centered_pow (power=4): [ 0.    4.01  4.94  5.    5.06  5.99 10.  ]


### 3.3 Own/custom function (sine)
Writing our own bin function is just as easy: it has to take `(low, high, n_bins)`
and return an array of `n_bins` edges *sorted in ascending order*, starting at `low` and
ending at `high`. `generate_box_bins` does not care how the edges in between were chosen --
any such callable works. Here we space the edges following a sine curve, which packs them
more densely near the two boundaries and more sparsely in the middle (the opposite of
`centered_pow_bin` above).

In [9]:
def sine_bin(low: float, high: float, n_bins: int) -> np.ndarray:
    """Bin edges spaced according to a sine curve: dense near the edges, sparse in the middle."""
    t = np.linspace(0, np.pi, n_bins)
    fraction = (1 - np.cos(t)) / 2  # monotonically 0 -> 1
    return low + fraction * (high - low)


sine_bins = generate_box_bins(one_dim_space, sine_bin, 7)
print("sine:          ", np.round(sine_bins[0], 2))

sine:           [ 0.    0.67  2.5   5.    7.5   9.33 10.  ]


### 3.4 Data-based bins
The bin functions above only look at the boundaries of the space.
But perhaps certain regions in the state-space are visited more frequently.
Given a dataset of samples (states or actions) we can create a discretization that evenly distributes those sampling into bins.
As a result, the space will be discretized into finer segments where there are many samples, and more coarse where there are less samples.
Because `bin_func` can be *any* `(low, high, n_bins) -> array` callable (section 3.3), a
data-driven bin function is just a closure over the dataset that returns its quantiles --
every bin then contains roughly the same number of samples.

In [10]:
rng = np.random.default_rng(0)
# bimodal data: more samples cluster around 2.0 and around 8.0
data = np.concatenate([rng.normal(2.0, 0.5, 200), rng.normal(8.0, 1.0, 200)])
data = np.clip(data, one_dim_space.low[0], one_dim_space.high[0])

quantile_bin_func = lambda low, high, n_bins: np.quantile(data, np.linspace(0, 1, n_bins))
data_bins = generate_box_bins(one_dim_space, quantile_bin_func, 7)
print("data-driven:   ", np.round(data_bins[0], 2))

data-driven:    [ 0.8   1.78  2.21  3.61  7.4   8.3  10.  ]


Compare this to the uniform `linspace` bins above: the edges are packed closer together
around 2.0 and 8.0, where `data` has more samples, and spread out where it has few.

A `BinEdges` object consists of three things, all still visible on the result above:

- `space`: the space it belongs to,
- `edges`: **all** bin edges of **all** dimensions in one flat array,
- `ranges`: for every dimension the `(start, end)` slice into `edges`.

### 3.5 Which one should I use?

| Discretization | How to get it | When to use it |
| --- | --- | --- |
| uniform | `np.linspace` | good default, no prior knowledge |
| polynomial | `centered_pow_bin` | interesting behaviour in the middle of the interval |
| custom | own bin function, e.g. `sine_bin` | when you know the shape you want analytically |
| data-based | a bin function built from quantiles of sampled data | we already have trajectories/observations |

Whatever we choose, the result is always a `BinEdges` object, and everything that follows
stays exactly the same.

## 4. From bin edges to an `AbstractionMap`

An `AbstractionMap` maps a *single* space (either state space or action space) between the original
and the abstract environment. Building one from a `BinEdges` used to mean assembling six
low-level functions by hand -- about 45 duplicated lines, both in `tests/utils.py` and in this
tutorial. It is now a single function call: `bin_edges_map(space, bin_edges)`.

The `forward_map` it builds stays **factored**: one bin index per dimension, e.g. `[2, 4]`,
a valid sample of `abstract_space` (a `MultiDiscrete`). We look at why that matters, and how
to get a single integer instead, in the next section.

In [11]:
from verigym.abstraction.abstractionmapper import bin_edges_map

factored_map = bin_edges_map(space, bin_edges)

print("abstract space:", factored_map.abstract_space)
print("forward: ", sample, "->", factored_map.forward_map(sample))
print(
    "backward:",
    factored_map.forward_map(sample),
    "->",
    factored_map.backward_map(factored_map.forward_map(sample)),
)

abstract space: MultiDiscrete([5 5])
forward:  [0.3 7.4] -> [2 2]
backward: [2 2] -> [0. 5.]


Note that the backward map does not return the original sample. That is expected:
the abstraction throws information away, so we only get the bin edge back that the sample was mapped to.
Also note that `has_backward_map` tells us whether a backward map was provided at all.

In [12]:
print("has backward map:", factored_map.has_backward_map)
print("original space is continuous:", factored_map.from_continuous_space)
print("elements original / abstract:", factored_map.original_n_elements, "/", factored_map.abstract_n_elements)

has backward map: True
original space is continuous: True
elements original / abstract: inf / 25


## 5. Factored output vs. `original_to_enum`

`forward_map` always stays factored (I), because that is what generalises to abstractions
that are not binnings at all (clustering, tile-coding, ...). But `create_abstraction` needs a
single `int` per abstract state -- it is used as a `dict` key and an array index
(`T_counts[s][a][s_next]`), and a factored `numpy` array is not hashable. `AbstractionMap`
gives you that single integer (E) too, without a second map:

- **`abstract_to_enum(idx)`** -- input is an *already-abstract* sample (I). Pure conversion, never
  touches `forward_map`.
- **`original_to_enum(x)`** -- input is an *original* sample (O). Does both steps at once:
  `abstract_to_enum(forward_map(x))`.

The pipeline always holds an original observation, so it calls `original_to_enum`. Code that
already has a factored abstract sample -- e.g. the `idx` we just printed above -- calls
`abstract_to_enum` directly.

In [13]:
idx = factored_map.forward_map(sample)  # O -> I  (factored, what section 4 showed)
enum_direct = factored_map.original_to_enum(sample)  # O -> I -> E, in one call
enum_from_idx = factored_map.abstract_to_enum(idx)  # I -> E, given an idx we already have

print("factored index (I):                       ", idx)
print("original_to_enum(sample)              (E):", enum_direct)
print("abstract_to_enum(forward_map(sample)) (E):", enum_from_idx)
assert enum_direct == enum_from_idx

factored index (I):                        [2 2]
original_to_enum(sample)              (E): 12
abstract_to_enum(forward_map(sample)) (E): 12


## 6. Combining two maps into an `AbstractionMapper`

An `AbstractionMapper` is simply the pair of maps for the *whole* environment:
one `AbstractionMap` for the states and one for the actions.
It offers accessors for both the factored and the enumerated direction:

- `original_to_abstract_state` / `original_to_abstract_state_enum`
- `abstract_to_original_state` (and the `_action` equivalents)

In [14]:
# The action space of our toy example: a 1-dimensional continuous action in [-2, 2].
action_space = gym.spaces.Box(low=-2.0, high=2.0, shape=(1,))
action_bin_edges = generate_box_bins(action_space, np.linspace, 5)
action_map = bin_edges_map(action_space, action_bin_edges)

mapper = AbstractionMapper(
    state_abstraction_map=factored_map,
    action_abstraction_map=action_map,
)

print("abstract states / actions:", mapper.abstract_n_states, "/", mapper.abstract_n_actions)
print("original states / actions:", mapper.original_n_states, "/", mapper.original_n_actions)
print("continuous states / actions:", mapper.from_continuous_states, "/", mapper.from_continuous_actions)

abstract states / actions: 25 / 5
original states / actions: inf / inf
continuous states / actions: True / True


In [15]:
state = np.array([-0.9, 2.5])
action = np.array([1.3])

abstract_state = mapper.original_to_abstract_state(state)
abstract_action = mapper.original_to_abstract_action(action)

print("state  ", state, "->", abstract_state, "->", mapper.abstract_to_original_state(abstract_state))
print("action ", action, "->", abstract_action, "->", mapper.abstract_to_original_action(abstract_action))
print("state  (enum)", state, "->", mapper.original_to_abstract_state_enum(state))
print("action (enum)", action, "->", mapper.original_to_abstract_action_enum(action))

state   [-0.9  2.5] -> [0 1] -> [-1.   2.5]
action  [1.3] -> [3] -> [1.]
state  (enum) [-0.9  2.5] -> 1
action (enum) [1.3] -> 3


If we did not provide a backward map, the backward direction raises a `ValueError`.

In [16]:
forward_only_map = AbstractionMap(
    forward_map=action_bin_edges.orig_to_idx,
    original_space=action_space,
    abstract_space=gym.spaces.MultiDiscrete(action_bin_edges.lengths),
)
forward_only_mapper = AbstractionMapper(factored_map, forward_only_map)

try:
    forward_only_mapper.abstract_to_original_action(0)
except ValueError as error:
    print("ValueError:", error)

ValueError: Cannot map abstract action to original action without a backward map in the action abstraction.


## 7. Backward kind: a point or a whole interval

The backward map we have used so far returns a single representative point per abstract
state -- handy when we need to execute *one* concrete action, e.g. when deploying a policy.
But model checking and state labeling need to know the *whole* region of the original space
that an abstract state stands for: "does the label `x > 2` hold in abstract state 17?" cannot
be answered from a single point.

`bin_edges_map(..., backward_kind="interval")` swaps the backward map for
`bin_edges.idx_to_interval`, which returns the lower and upper bound per dimension instead of
one representative value. The map records which kind it returns in `backward_kind`, so code
that only sees an `AbstractionMap` can branch on it instead of guessing (a guess is what
caused a real bug in the labeling code this abstraction layer feeds -- see
`AbstractStateLabeler`).

In [17]:
interval_map = bin_edges_map(space, bin_edges, backward_kind="interval")
print("factored_map.backward_kind:", factored_map.backward_kind)
print("interval_map.backward_kind:", interval_map.backward_kind)

idx = interval_map.forward_map(sample)
print("point    backward:", factored_map.backward_map(idx))
print("interval backward:\n", interval_map.backward_map(idx))

factored_map.backward_kind: point
interval_map.backward_kind: interval
point    backward: [0. 5.]
interval backward:
 [[0.  5. ]
 [0.5 7.5]]


**Honesty check.** Because `n_bins` counts *representative values*, not cells, the very top
interval along each dimension is degenerate: index `n_bins - 1` is reached only by a sample
that lands exactly on `space.high`, so its interval collapses to `[high, high]`. Every other
interval is a normal, non-empty cell.

In [18]:
top_idx = bin_edges.orig_to_idx(space.high)
print("top idx:            ", top_idx)
print("its interval:\n", interval_map.backward_map(top_idx))

top idx:             [4 4]
its interval:
 [[ 1. 10.]
 [ 1. 10.]]


## 8. Identity mapping

Sometimes no abstraction is needed, because the environment is already discrete
(e.g. `Taxi-v3`). Then we use the identity map, which returns every input unchanged.

In [19]:
identity_mapper = AbstractionMapper.initialize_identity_mapper(
    state_space=gym.spaces.Discrete(500),
    action_space=gym.spaces.Discrete(6),
)

print("state 3 ->", identity_mapper.original_to_abstract_state(3))
print("action 1 ->", identity_mapper.original_to_abstract_action(1))
print("abstract states / actions:", identity_mapper.abstract_n_states, "/", identity_mapper.abstract_n_actions)

state 3 -> 3
action 1 -> 1
abstract states / actions: 500 / 6


A single map can be built the same way with `AbstractionMap.initialize_identity_map(space)`.
This is handy when only one of the two spaces needs to be abstracted, e.g. continuous states
but already discrete actions.

## 9. Putting it all together on a real environment

Finally we build an `AbstractionMapper` for `CartPole-v1`, which is what we hand over to
`create_abstraction`. Two practical details show up here:

1. The observation space of `CartPole-v1` is unbounded (`inf`). We cannot cut an infinite
   interval into bins, so we first replace the infinities with `ReplaceInfObservation`.
2. The action space is `gym.spaces.Discrete(2)`, so an action is a plain `int`, not an array.
   `BinEdges`' codec methods handle this transparently -- they promote scalars to arrays
   internally and cast the result back to a valid sample of the original space.

Building the mapper for both spaces used to require a ~45-line helper function
(`build_abstraction_mapper`, hand-assembled from six low-level functions and duplicated in
`tests/utils.py`), plus a separate `CachedDiscretizer` wrapped around the hot path. All of
that collapses into a single `binned_mapper(...)` call -- caching included, via `cache=True`.

In [20]:
from verigym.abstraction.abstractionmapper import binned_mapper
from verigym.abstraction.gym_utils.transform_observation import ReplaceInfObservation

env = gym.make("CartPole-v1")
env = ReplaceInfObservation(env, neg_inf=-10, pos_inf=10)

cartpole_mapper = binned_mapper(env, np.linspace, n_bins_states=5, n_bins_actions=2, cache=True)
print("abstract states / actions:", cartpole_mapper.abstract_n_states, "/", cartpole_mapper.abstract_n_actions)

observation, _info = env.reset(seed=42)
print("observation", observation, "-> enum", cartpole_mapper.original_to_abstract_state_enum(observation))
print("action     ", 1, "-> enum", cartpole_mapper.original_to_abstract_action_enum(1))

abstract states / actions: 625 / 2
observation [ 0.0273956  -0.00611216  0.03585979  0.0197368 ] -> enum 287


action      1 -> enum 1


Changing the discretization is now a one-line change: we pass a different `bin_func`.
The number of abstract states stays the same, but *where* the states lie changes,
and therefore observations can end up in a different abstract state.

In [21]:
pow_mapper = binned_mapper(env, centered_pow_bin, n_bins_states=5, n_bins_actions=2, cache=True)

env.observation_space.seed(0)
observations = [env.observation_space.sample() for _ in range(5)]

print(f"{'observation':>44} | linspace | polynomial")
for observation in observations:
    print(
        f"{str(np.round(observation, 2)):>44} |"
        f"{cartpole_mapper.original_to_abstract_state_enum(observation):>9} |"
        f"{pow_mapper.original_to_abstract_state_enum(observation):>11}"
    )

                                 observation | linspace | polynomial


                   [ 1.31 -4.6  -0.38 -9.67] |      275 |        375
                       [3.01 8.26 0.09 4.59] |      462 |        463
                   [ 0.42  8.7   0.26 -9.95] |      340 |        340
                   [ 3.43 -9.33  0.19 -6.49] |      385 |        390
                   [ 3.49  0.83 -0.17 -1.55] |      431 |        426


The resulting `AbstractionMapper` is all that `create_abstraction` needs to turn the
simulator into an explicit MDP. This used to be structurally impossible for a hand-built
factored map (`forward_map` returning an array raises `TypeError: unhashable type:
'numpy.ndarray'` the moment it is used as a dict key) -- `original_to_enum` is what fixes it.

In [22]:
from verigym.policy.policy import RandomizedPolicy

generative_env = verigym.GenerativeEnv.from_gymnasium(env)
abstracted_env = verigym.create_abstraction(
    original_env=generative_env,
    abstraction_mapper=cartpole_mapper,
    exploration_policy=RandomizedPolicy(generative_env),
    num_steps=1000,
    multithreading=False,
)
print("number of abstract states reached:", abstracted_env.nr_states)

Simulation time: 0.0181s
Trajectories in dataset: 46
Learning Abstraction: 0.0090s
number of abstract states reached: 625


## Summary

- `BinEdges` describes *where* a continuous space is cut; create it with
  `generate_box_bins(space, bin_func, n_bins)`.
- The `bin_func` decides the *type* of discretization: uniform (`np.linspace`), polynomial
  (`centered_pow_bin`), any custom `(low, high, n_bins) -> array` callable (e.g. `sine_bin`),
  or data-driven (a closure returning quantiles of sampled observations).
- `bin_edges_map(space, bin_edges)` builds an `AbstractionMap` in a single call -- no more
  hand-assembling six low-level functions.
- `forward_map` always stays **factored** (one bin index per dimension). `abstract_to_enum` /
  `original_to_enum` give the single `int` that `create_abstraction` needs: the first takes an
  already-abstract sample, the second takes an original sample and does both steps.
- `backward_kind` (`"point"` | `"interval"` | ...) declares what the backward map returns. Use
  `"interval"` when you need the whole cell an abstract state stands for (model checking,
  labeling) rather than one representative point -- and remember the very top interval per
  dimension is degenerate (`[high, high]`), since `n_bins` counts representative values, not cells.
- `binned_mapper(env, bin_func, n_bins_states, n_bins_actions)` builds a full
  `AbstractionMapper` for both state and action spaces in one call -- what used to be a
  ~45-line helper function assembled from six low-level primitives.
- Use `AbstractionMapper.initialize_identity_mapper` when a space is already discrete and
  needs no abstraction.
- The resulting `AbstractionMapper` is all `create_abstraction` needs to turn a simulator into
  an explicit MDP.